**Load All Data Files (EEG Data, Metadata, and Configuration Files)**

In [ ]:
import os
import openneuro

# Define BIDS root folder
bids_root = "PD3"
os.makedirs(bids_root, exist_ok=True)

# Download dataset (all subjects, all sessions)
openneuro.download(dataset="ds003506", target_dir=bids_root)

print("✅ Download complete.")

In [ ]:
import pandas as pd
import os

# Paths
bids_root = "PD3"
participants_tsv = os.path.join(bids_root, "56participants.tsv")
metadata_xlsx = os.path.join(bids_root, "code", "56PDDys_4BIDS.xlsx")

# Load participant info
participants_df = pd.read_csv(participants_tsv, sep='\t')
print("✅ Loaded 56participants.tsv")

# Optional: load extra metadata
extra_metadata_df = pd.read_excel(metadata_xlsx)
print("✅ Loaded PDDys_4BIDS.xlsx")

# Preview
print(participants_df.head())
print(extra_metadata_df.head())


**Step 1: Preprocess and Extract Features for Each Subject**

In [ ]:
from mne_bids import get_entity_vals
import os

# Get list of all subjects
all_subjects = get_entity_vals(bids_root, entity_key='subject')
print(f"✅ Found {len(all_subjects)} subjects: {all_subjects}")

# Check session(s) per subject
def find_sessions(subject):
    subject_dir = os.path.join(bids_root, f"sub-{subject}")
    return [s.replace("ses-", "") for s in os.listdir(subject_dir) if s.startswith("ses-")]

subject_sessions = {sub: find_sessions(sub) for sub in all_subjects}
print("✅ Sessions per subject:", subject_sessions)



**Step 2: Loop Over All Subjects**

In [ ]:
import os
import mne

bids_root = "PD3"  # make sure this is defined

def load_eeg(subject, session):
    eeg_dir = os.path.join(bids_root, f"sub-{subject}", f"ses-{session}", "eeg")
    print(f"🔎 Searching in {eeg_dir}")

    if not os.path.exists(eeg_dir):
        print(f"❌ Directory not found: {eeg_dir}")
        return None

    print("📁 Files in directory:")
    for file in os.listdir(eeg_dir):
        print("  ", file)

    for file in os.listdir(eeg_dir):
        if file.endswith("_eeg.set"):
            eeg_file = os.path.join(eeg_dir, file)
            print(f"📥 Loading EEG: {eeg_file}")
            try:
                raw = mne.io.read_raw_eeglab(eeg_file, preload=True)
                raw.filter(1., 40., fir_design='firwin')
                return raw
            except Exception as e:
                print(f"❌ Failed to load {eeg_file}: {e}")
                return None

    print(f"❌ No EEG .set file found in {eeg_dir}")
    return None

# ✅ Call the function OUTSIDE
raw = load_eeg('001', '01')


In [ ]:
from mne.time_frequency import psd_array_welch
import numpy as np
import pandas as pd

def extract_features(raw, bands):
    data = raw.get_data()
    sf = raw.info['sfreq']

    psds, freqs = psd_array_welch(data, sf, n_fft=256, verbose=False)
    psds_db = 10 * np.log10(psds)

    features = []
    for band in bands:
        fmin, fmax = band
        idx = np.logical_and(freqs >= fmin, freqs <= fmax)
        band_power = psds_db[:, idx].mean(axis=1)
        features.extend(band_power)

    for ch in data:
        features.append(np.mean(ch))
        features.append(np.std(ch))
        features.append(pd.Series(ch).skew())
        features.append(pd.Series(ch).kurtosis())
    
    return np.array(features)

bands = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
features = extract_features(raw, bands)

print(f"✅ Extracted {len(features)} features.")


In [ ]:
import os
import numpy as np
import pandas as pd
import mne
from mne_bids import get_entity_vals

# === Define Band Ranges ===
bands = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30)
}

# === Feature Extraction Function ===
def extract_features(epochs, bands):
    n_times = epochs.get_data().shape[-1]
    n_fft = min(256, n_times)

    psd = epochs.compute_psd(method="welch", fmin=1, fmax=40, n_fft=n_fft)
    freqs = psd.freqs
    psd_data = psd.get_data()

    feature_vector = []
    for band_name, (fmin, fmax) in bands.items():
        idx_band = (freqs >= fmin) & (freqs <= fmax)
        band_power = psd_data[:, :, idx_band].mean(axis=2)
        feature_vector.append(band_power)

    features = np.concatenate(feature_vector, axis=1)
    return features

# === EEG Loader Function ===
def load_eeg(subj, ses, retry=False):
    eeg_file = f"PD3/sub-{subj}/ses-{ses}/eeg/sub-{subj}_ses-{ses}_task-ReinforcementLearning_eeg.set"

    try:
        raw = mne.io.read_raw_eeglab(eeg_file, preload=True)

        # 💡 Crop beginning of raw to skip unstable data
        raw.crop(tmin=400.0)

        # 💡 Remove 'boundary' annotations
        raw.annotations.delete(np.where(raw.annotations.description == 'boundary')[0])

        # 💡 Now filter safely
        raw = raw.filter(1., 40., fir_design='firwin')

        # ✅ Extract events excluding 'boundary'
        try:
            events, event_id = mne.events_from_annotations(raw)
            event_id = {k: v for k, v in event_id.items() if 'boundary' not in k.lower()}
            if len(event_id) == 0:
                raise ValueError("No valid events after removing 'boundary'")
        except Exception as e_ann:
            if retry:
                print(f"🔄 Retrying with stim channel for sub-{subj}_ses-{ses}")
                events = mne.find_events(raw)
                if len(events) == 0:
                    raise ValueError("No events from stim channels.")
                event_id = dict(zip(np.unique(events[:, 2]), np.unique(events[:, 2])))
            else:
                raise RuntimeError(f"No usable events for sub-{subj}_ses-{ses}: {e_ann}")

        # 💡 Epoch safely with annotation rejection enabled
        epochs = mne.Epochs(
            raw, events, event_id=event_id,
            tmin=-0.2, tmax=0.5,
            baseline=(-0.2, 0),
            preload=True,
            reject_by_annotation=True
        )

        if len(epochs) == 0:
            raise RuntimeError(f"No epochs found for sub-{subj}_ses-{ses}")

        # Drop epochs with any NaNs or Infs
        data = epochs.get_data()
        is_finite = np.isfinite(data).all(axis=(1, 2))
        epochs = epochs[is_finite]

        if len(epochs) == 0:
            raise RuntimeError(f"All epochs contain NaNs/Infs for sub-{subj}_ses-{ses}")

        return epochs

    except Exception as e:
        print(f"❌ Failed to load or epoch sub-{subj} ses-{ses}: {e}")
        return None


# === Main Script ===
bids_root = "PD3"
participants_path = os.path.join(bids_root, "56participants.tsv")
participants_df = pd.read_csv(participants_path, sep="\t")

subject_to_label = {
    row['participant_id'].split('-')[-1]: 1 if row['Group'].strip().upper() == 'PD' else 0
    for _, row in participants_df.iterrows()
}

all_subjects = get_entity_vals(bids_root, entity_key='subject')
print(f"🔍 Total subjects in dataset: {len(all_subjects)}")

X_all, y_all, subject_ids = [], [], []
failed_sessions, successful_sessions, successful_shapes = [], [], []

# === Loop Through Subjects & Sessions ===
for subj in all_subjects:
    if subj not in subject_to_label:
        print(f"⚠️ Skipping {subj} (not in participants.tsv)")
        continue

    label = subject_to_label[subj]
    n_sessions = 2 if label == 1 else 1

    for ses_num in range(1, n_sessions + 1):
        ses = f"{ses_num:02d}"
        session_key = f"{subj}_ses-{ses}"
        print(f"🔍 Processing {session_key}")

        try:
            epochs = load_eeg(subj, ses)
            if epochs is None:
                failed_sessions.append(session_key)
                continue

            features = extract_features(epochs, bands)
            X_all.append(features)
            y_all.append(label)
            subject_ids.append(session_key)
            successful_sessions.append(session_key)
            successful_shapes.append(features.shape)
            print(f"✅ {session_key} | Features: {len(features)} | Shape: {features.shape}")

        except Exception as e:
            print(f"❌ Feature extraction failed for {session_key}: {e}")
            if "No usable events" in str(e) or "annotations" in str(e).lower():
                print(f"🔄 Retrying {session_key} with retry logic")
                epochs = load_eeg(subj, ses, retry=True)
                if epochs is None:
                    failed_sessions.append(session_key)
                    continue
                features = extract_features(epochs, bands)
                X_all.append(features)
                y_all.append(label)
                subject_ids.append(session_key)
                successful_sessions.append(session_key)
                successful_shapes.append(features.shape)
                print(f"✅ {session_key} | Features (after retry): {len(features)} | Shape: {features.shape}")
            else:
                failed_sessions.append(session_key)

# === Summary ===
print("\n📉 Failed Sessions:")
for f in failed_sessions:
    print(" -", f)

print(f"\n✅ Successful Sessions:")
for s, shape in zip(successful_sessions, successful_shapes):
    print(f" - {s} | Shape: {shape}")

print(f"\n❗ Total failed sessions: {len(failed_sessions)}")
print(f"✅ Total successful sessions: {len(successful_sessions)}")

total_samples = sum([shape[0] for shape in successful_shapes])
print(f"\n📊 Total samples processed: {total_samples}")



In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

# Filter out unsuccessful subjects
X_filtered, y_filtered, subject_ids_filtered = [], [], []

# Assuming successful_subjects is a list of subjects that were processed successfully
successful_subjects = successful_sessions  # Use the successful_sessions list you already have

for session_str, label in zip(subject_ids, y_all):
    if session_str in successful_subjects:
        # Get the index for the corresponding features
        index = subject_ids.index(session_str)
        X_filtered.append(X_all[index])
        
        # Repeat the label for each epoch in the session
        n_epochs = X_all[index].shape[0]
        y_filtered.extend([label] * n_epochs)
        
        # Repeat the subject_id for each epoch
        subject_ids_filtered.extend([session_str] * n_epochs)

# Convert the list of arrays into a 2D array (if all arrays have the same shape)
X_filtered = np.vstack(X_filtered)  # Stacking the features vertically
y_filtered = np.array(y_filtered)

print(f"\n📊 Filtered data shape: {X_filtered.shape}, Labels shape: {y_filtered.shape}")
print(f"🔢 Class distribution (filtered data): {Counter(y_filtered)}")

# ✅ Save filtered data to a .npz file
np.savez("eeg_features_filtered_PD3.npz", X=X_filtered, y=y_filtered, subjects=subject_ids_filtered)
print("✅ Saved filtered data as eeg_features_filtered_PD3.npz")

# ✅ Or save filtered data to a .csv file
df_filtered = pd.DataFrame(X_filtered)  # Convert to DataFrame for easy saving
df_filtered["label"] = y_filtered
df_filtered["subject_id"] = subject_ids_filtered

df_filtered.to_csv("eeg_features_filtered_PD3.csv", index=False)
print("✅ Saved filtered data as eeg_features_filtered_PD3.csv")



**Step 3: Split Data into Training and Testing Sets**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

# Step 1: Get unique subject-label pairs
unique_subjects = list(set(subject_ids_filtered))
subject_to_label = {subj: y_filtered[np.where(np.array(subject_ids_filtered) == subj)[0][0]]
                    for subj in unique_subjects}
subjects = np.array(list(subject_to_label.keys()))
labels = np.array(list(subject_to_label.values()))

# Step 2: Subject-level stratified split
train_subjects, test_subjects = train_test_split(
    subjects, test_size=0.2, stratify=labels, random_state=42
)

# Step 3: Extract epoch-level data based on subject splits
X_train, y_train, X_test, y_test = [], [], [], []

for X_epoch, label, subj in zip(X_filtered, y_filtered, subject_ids_filtered):
    if subj in train_subjects:
        X_train.append(X_epoch)
        y_train.append(label)
    elif subj in test_subjects:
        X_test.append(X_epoch)
        y_test.append(label)

X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

# Print confirmation
print(f"📊 Training data shape: {X_train.shape}, Labels shape: {y_train.shape}")
print(f"📊 Testing data shape: {X_test.shape}, Labels shape: {y_test.shape}")
print(f"🔢 Train class distribution: {Counter(y_train)}")
print(f"🔢 Test class distribution: {Counter(y_test)}")


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import os
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

# Path to the root directory where your subjects and sessions are stored
root_dir = "PD3"

# Subject-to-label mapping
subject_to_label = {}

# Add PD subjects (sub-002 to sub-031)
for subj in range(2, 32):
    subject_to_label[f'sub-{str(subj).zfill(3)}'] = 1  # PD subjects

# Add CTL subjects (sub-001, sub-007, sub-009, ..., sub-056)
for subj in [1, 7, 9] + list(range(32, 57)):
    subject_to_label[f'sub-{str(subj).zfill(3)}'] = 0  # Control subjects

# Initialize lists to store data
X_all, y_all, session_labels = [], [], []

# Function to load EEG data from the session folder (this will depend on your EEG data format)
def load_eeg_data(subject_id, session_id):
    eeg_dir = os.path.join(root_dir, subject_id, session_id, "eeg")
    
    # Assuming you have EEG data files in .fdt, .set, or another format
    # Adjust this function according to the actual data format
    eeg_file = os.path.join(eeg_dir, "eeg_data.set")  # Example file name
    
    # Here you should load the data from the eeg_file
    # This is just a placeholder; replace with your actual data loading method
    # For example, using MNE:
    # eeg_data = mne.io.read_raw_eeglab(eeg_file, preload=True)
    
    # Placeholder: return random data for now
    n_samples = 100  # Assume 100 samples for each session (adjust as needed)
    n_features = 268  # Number of features (e.g., channels, frequency bands)
    eeg_data = np.random.rand(n_samples, n_features)  # Random data placeholder
    
    return eeg_data

# Step 1: Iterate through subjects and sessions
for subj in subject_to_label:
    subject_label = subject_to_label[subj]  # 1 for PD, 0 for control
    
    # For PD subjects, we have two sessions: ses-01, ses-02
    if subject_label == 1:  # PD subjects
        sessions = ['ses-01', 'ses-02']
    else:  # Control subjects, we have only one session: ses-01
        sessions = ['ses-01']
    
    # Iterate through sessions for the current subject
    for ses in sessions:
        # Load the EEG data for the session
        X_session = load_eeg_data(subj, ses)  # This will return the features (e.g., EEG data)
        
        # Append the data to X_all and the label to y_all
        X_all.append(X_session)
        y_all.extend([subject_label] * X_session.shape[0])  # Repeat the label for each sample in the session
        session_labels.extend([ses] * X_session.shape[0])  # Add session information (e.g., 'ses-01', 'ses-02')

# Convert lists into numpy arrays
X_all = np.vstack(X_all)  # Stack the features vertically to create the full dataset
y_all = np.array(y_all)   # Convert labels to numpy array

# Step 2: Verify the data
print(f"📊 Data shape: {X_all.shape}, Labels shape: {y_all.shape}")
print(f"🔢 Class distribution: {Counter(y_all)}")

# Combine label and session info into one array for stratification
stratify_labels = [f"{label}_{session}" for label, session in zip(y_all, session_labels)]

# Step 3: Perform stratified split based on session labels (if you want to stratify by both class and session)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    stratify= stratify_labels ,  # Stratify by both class and session
    random_state=42 )

# Step 4: Print the shapes and class distributions after splitting
print(f"📊 Training data shape: {X_train.shape}, Labels shape: {y_train.shape}")
print(f"📊 Testing data shape: {X_test.shape}, Labels shape: {y_test.shape}")
print(f"🔢 Train class distribution: {Counter(y_train)}")
print(f"🔢 Test class distribution: {Counter(y_test)}")


In [ ]:
combined_stratify_labels = [f"{label}_{session}" for label, session in zip(y_all, session_labels)]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    stratify=combined_stratify_labels,  # Stratify by combined class+session labels
    random_state=42
)

print(f"📊 Training data shape: {X_train.shape}, Labels shape: {y_train.shape}")
print(f"📊 Testing data shape: {X_test.shape}, Labels shape: {y_test.shape}")
print(f"🔢 Train class distribution: {Counter(y_train)}")
print(f"🔢 Test class distribution: {Counter(y_test)}")

**Step 4: Train ML Classifiers**

In [ ]:
num_subjects = 56
epochs_per_subject = len(y) // num_subjects
subject_ids = np.repeat(np.arange(num_subjects), epochs_per_subject)

In [ ]:
# Example: subject_ids = np.array([...])  # shape = (57659,)
np.savez("eeg_features_filtered_PD3_with_ids.npz", X=X, y=y, subject_ids=subject_ids)

**Randon Forest**

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats


# Initialize the RandomForestClassifier with class weights to handle imbalance
rf_classifier = RandomForestClassifier(random_state=42, class_weight='balanced')

# Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store results for each fold
precision_list = []
recall_list = []
f1_list = []
accuracy_list = []

# Loop through each fold
for fold, (train_idx, test_idx) in enumerate(skf.split(X_filtered, y_filtered), 1):
    print(f"Training Fold {fold}...")

    # Split the data into train and test sets for this fold
    X_train, X_test = X_filtered[train_idx], X_filtered[test_idx]
    y_train, y_test = y_filtered[train_idx], y_filtered[test_idx]

    # Check for NaN or infinite values in X_train and X_test
    if np.any(np.isnan(X_train)) or np.any(np.isnan(X_test)):
        print("Warning: NaN values found in the data")
        
    if np.any(np.isinf(X_train)) or np.any(np.isinf(X_test)):
        print("Warning: Infinite values found in the data")

    # Replace infinite values with a large finite number (or NaN)
    X_train = np.nan_to_num(X_train, nan=0, posinf=1e10, neginf=-1e10)
    X_test = np.nan_to_num(X_test, nan=0, posinf=1e10, neginf=-1e10)

    # Check for outliers using Z-scores (Z > 3 is typically considered an outlier)
    z_scores = np.abs(stats.zscore(X_train))
    X_train = X_train[(z_scores < 3).all(axis=1)]
    y_train = y_train[(z_scores < 3).all(axis=1)]

    # Similarly for X_test
    z_scores_test = np.abs(stats.zscore(X_test))
    X_test = X_test[(z_scores_test < 3).all(axis=1)]
    y_test = y_test[(z_scores_test < 3).all(axis=1)]

    # Scale the features using StandardScaler
    scaler = StandardScaler()

    # Fit and transform the training data
    X_train_scaled = scaler.fit_transform(X_train)

    # Transform the test data
    X_test_scaled = scaler.transform(X_test)

    # Train the model
    rf_classifier.fit(X_train_scaled, y_train)

    # Make predictions
    y_pred = rf_classifier.predict(X_test_scaled)

    # Calculate precision, recall, and F1-score for the current fold
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
    accuracy = np.mean(y_pred == y_test)

    # Append to the lists
    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    accuracy_list.append(accuracy)

    # Print classification report for the fold
    print(f"\nClassification Report (Fold {fold}):")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix (Fold {fold})')
    plt.tight_layout()  # Ensures no clipping of labels
    plt.show()

# Aggregate results after all folds
print("\nAggregated Performance Metrics:")
print(f"Mean Accuracy: {np.mean(accuracy_list):.4f} ± {np.std(accuracy_list):.4f}")
print(f"Mean Precision: {np.mean(precision_list):.4f} ± {np.std(precision_list):.4f}")
print(f"Mean Recall: {np.mean(recall_list):.4f} ± {np.std(recall_list):.4f}")
print(f"Mean F1-score: {np.mean(f1_list):.4f} ± {np.std(f1_list):.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from scipy import stats
from collections import Counter

# -------------------------------
# Replace with your data loading
# -------------------------------
data = np.load("eeg_features_filtered_PD3_with_ids.npz")
X = data["X"]
y = data["y"]
subject_ids = data["subject_ids"]


# -------------------------------
# Parameters
# -------------------------------
n_splits = 5
top_n_features = 50  # Based on feature importance
random_state = 42

# -------------------------------
# Model Setup
# -------------------------------
rf_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=random_state
)

# -------------------------------
# Cross-validation setup
# -------------------------------
gkf = GroupKFold(n_splits=n_splits)

# Performance lists
precision_list, recall_list, f1_list, accuracy_list = [], [], [], []

# -------------------------------
# Cross-validation loop
# -------------------------------
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=subject_ids), 1):
    print(f"\n📁 Fold {fold}")

    # Split sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    subj_train, subj_test = subject_ids[train_idx], subject_ids[test_idx]

    # Handle NaNs/Infs
    X_train = np.nan_to_num(X_train, nan=0, posinf=1e10, neginf=-1e10)
    X_test = np.nan_to_num(X_test, nan=0, posinf=1e10, neginf=-1e10)

    # Z-score outlier removal
    z_scores = np.abs(stats.zscore(X_train))
    non_outlier_mask = (z_scores < 3).all(axis=1)
    X_train, y_train, subj_train = X_train[non_outlier_mask], y_train[non_outlier_mask], subj_train[non_outlier_mask]

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Fit model
    rf_classifier.fit(X_train_scaled, y_train)

    # Select top N features based on importance
    importances = rf_classifier.feature_importances_
    top_indices = np.argsort(importances)[-top_n_features:]
    X_train_selected = X_train_scaled[:, top_indices]
    X_test_selected = X_test_scaled[:, top_indices]

    # Re-train with selected features
    rf_classifier.fit(X_train_selected, y_train)

    # Predict
    y_pred_epoch = rf_classifier.predict(X_test_selected)

    # Majority vote by subject
    def majority_vote(preds, labels, subjects):
        subject_predictions = {}
        subject_true = {}
        unique_subjs = np.unique(subjects)
        for subj in unique_subjs:
            mask = (subjects == subj)
            votes = preds[mask]
            majority = Counter(votes).most_common(1)[0][0]
            subject_predictions[subj] = majority
            subject_true[subj] = Counter(labels[mask]).most_common(1)[0][0]  # True label
        return list(subject_predictions.values()), list(subject_true.values())

    y_pred_subj, y_true_subj = majority_vote(y_pred_epoch, y_test, subj_test)

    # Metrics
    precision, recall, f1, _ = precision_recall_fscore_support(y_true_subj, y_pred_subj, average='binary')
    accuracy = np.mean(np.array(y_pred_subj) == np.array(y_true_subj))

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    accuracy_list.append(accuracy)

    # Report
    print(classification_report(y_true_subj, y_pred_subj, target_names=['Control', 'PD']))
    cm = confusion_matrix(y_true_subj, y_pred_subj)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
    plt.title(f'Confusion Matrix (Fold {fold})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

# -------------------------------
# Final Summary
# -------------------------------
print("\n📊 Final Cross-Validation Results:")
print(f"Mean Accuracy:  {np.mean(accuracy_list):.4f} ± {np.std(accuracy_list):.4f}")
print(f"Mean Precision: {np.mean(precision_list):.4f} ± {np.std(precision_list):.4f}")
print(f"Mean Recall:    {np.mean(recall_list):.4f} ± {np.std(recall_list):.4f}")
print(f"Mean F1-score:  {np.mean(f1_list):.4f} ± {np.std(f1_list):.4f}")


**SVM**

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Prepare unique subject-label mapping (assuming you have subject_ids_filtered)
unique_subjects = np.unique(subject_ids_filtered)
subject_to_label = {subj: y_filtered[np.where(np.array(subject_ids_filtered) == subj)[0][0]] for subj in unique_subjects}

subjects = np.array(list(subject_to_label.keys()))
subject_labels = np.array(list(subject_to_label.values()))

# Initialize StratifiedKFold for subjects
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_classification_reports = []
all_confusion_matrices = []

# For averaging metrics
precision_list = []
recall_list = []
fscore_list = []

for fold, (train_subj_idx, test_subj_idx) in enumerate(skf.split(subjects, subject_labels), 1):
    print(f"Training Fold {fold}...")

    # Get train/test subjects
    train_subjects = subjects[train_subj_idx]
    test_subjects = subjects[test_subj_idx]

    # Select epochs for train/test based on subject ids
    train_indices = [i for i, subj in enumerate(subject_ids_filtered) if subj in train_subjects]
    test_indices = [i for i, subj in enumerate(subject_ids_filtered) if subj in test_subjects]

    X_train, y_train = X_filtered[train_indices], y_filtered[train_indices]
    X_test, y_test = X_filtered[test_indices], y_filtered[test_indices]

    # Pipeline: Imputer -> Scaler -> SVM
    svm_pipeline = make_pipeline(
        SimpleImputer(strategy='mean'),
        StandardScaler(),
        SVC(random_state=42, class_weight='balanced')
    )

    # Train the model
    svm_pipeline.fit(X_train, y_train)

    # Predict on test set
    y_pred = svm_pipeline.predict(X_test)

    # Classification report & confusion matrix
    print(f"\nClassification Report (SVM) for Fold {fold}:")
    report = classification_report(y_test, y_pred)
    print(report)
    all_classification_reports.append(report)

    cm = confusion_matrix(y_test, y_pred)
    all_confusion_matrices.append(cm)

    # Plot confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix (SVM) - Fold {fold}')
    plt.tight_layout()
    plt.show()

    # Store precision, recall, f1-score for averaging
    precision, recall, fscore, _ = precision_recall_fscore_support(
        y_test, y_pred, average=None, labels=[0, 1]
    )
    precision_list.append(precision)
    recall_list.append(recall)
    fscore_list.append(fscore)

# Average metrics across folds
avg_precision = np.mean(precision_list, axis=0)
avg_recall = np.mean(recall_list, axis=0)
avg_fscore = np.mean(fscore_list, axis=0)

print("\n=== Average Metrics Across Folds ===")
print(f"Precision (Control, PD): {avg_precision}")
print(f"Recall    (Control, PD): {avg_recall}")
print(f"F1-Score  (Control, PD): {avg_fscore}")



In [ ]:
import numpy as np

# Store results for aggregated metrics
precision_class_0 = []
recall_class_0 = []
f1_class_0 = []

precision_class_1 = []
recall_class_1 = []
f1_class_1 = []

accuracy_list = []

# Now reports are dictionaries
for report in all_classification_reports:
    precision_class_0.append(report["0"]["precision"])
    recall_class_0.append(report["0"]["recall"])
    f1_class_0.append(report["0"]["f1-score"])
    
    precision_class_1.append(report["1"]["precision"])
    recall_class_1.append(report["1"]["recall"])
    f1_class_1.append(report["1"]["f1-score"])
    
    accuracy_list.append(report["accuracy"])

# Aggregate performance metrics
print(f"\n📈 Aggregated Cross-Validation Results:")
print(f"Class 0 (Control)             - Precision: {np.mean(precision_class_0):.4f}, Recall: {np.mean(recall_class_0):.4f}, F1: {np.mean(f1_class_0):.4f}")
print(f"Class 1 (Parkinson's Disease) - Precision: {np.mean(precision_class_1):.4f}, Recall: {np.mean(recall_class_1):.4f}, F1: {np.mean(f1_class_1):.4f}")
print(f"Overall Accuracy              : {np.mean(accuracy_list):.4f}")


**Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

# Stratified K-Fold Cross-Validation to ensure class distribution
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize the lists for storing results
all_classification_reports = []
all_confusion_matrices = []

# Loop through each fold
for fold, (train_idx, test_idx) in enumerate(skf.split(X_filtered, y_filtered), 1):
    print(f"Training Fold {fold}...")

    # Split the data into train and test sets for this fold
    X_train, X_test = X_filtered[train_idx], X_filtered[test_idx]
    y_train, y_test = y_filtered[train_idx], y_filtered[test_idx]

    # Create a pipeline: Impute missing values -> Standardize data -> Train Logistic Regression
    logreg_pipeline = make_pipeline(
        SimpleImputer(strategy='mean'),  # Handle missing data
        StandardScaler(),                # Standardize features
        LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)  # Logistic Regression Classifier
    )

    # Train the model using the pipeline
    logreg_pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred_logreg = logreg_pipeline.predict(X_test)

    # Print evaluation metrics
    print(f"\nClassification Report (Logistic Regression) for Fold {fold}:")
    class_report = classification_report(y_test, y_pred_logreg, output_dict=True)
    print(class_report)
    all_classification_reports.append(class_report)

    # Confusion Matrix
    cm_logreg = confusion_matrix(y_test, y_pred_logreg)
    all_confusion_matrices.append(cm_logreg)

    # Plot confusion matrix
    sns.heatmap(cm_logreg, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix (Logistic Regression) - Fold {fold}')
    plt.tight_layout()  # Ensures no clipping of labels
    plt.show()

# Aggregate results after all folds
print("\nAggregated Classification Reports:")
for i, report in enumerate(all_classification_reports, 1):
    print(f"Fold {i} Classification Report:\n{report}")

# Optionally, calculate and print average accuracy, precision, recall, f1-score
# for each class across all folds
import numpy as np

# Aggregation lists
precision_class_0 = []
recall_class_0 = []
f1_class_0 = []

precision_class_1 = []
recall_class_1 = []
f1_class_1 = []

accuracy_list = []

# Loop over reports
for report in all_classification_reports:
    precision_class_0.append(report["0"]["precision"])
    recall_class_0.append(report["0"]["recall"])
    f1_class_0.append(report["0"]["f1-score"])
    
    precision_class_1.append(report["1"]["precision"])
    recall_class_1.append(report["1"]["recall"])
    f1_class_1.append(report["1"]["f1-score"])
    
    accuracy_list.append(report["accuracy"])

# Print aggregated results
print("\n📈 Aggregated Cross-Validation Results (Logistic Regression):")
print(f"Class 0 (Control)             - Precision: {np.mean(precision_class_0):.4f}, Recall: {np.mean(recall_class_0):.4f}, F1: {np.mean(f1_class_0):.4f}")
print(f"Class 1 (Parkinson's Disease) - Precision: {np.mean(precision_class_1):.4f}, Recall: {np.mean(recall_class_1):.4f}, F1: {np.mean(f1_class_1):.4f}")
print(f"Overall Accuracy              : {np.mean(accuracy_list):.4f}")


**Step 5: Transformer Model**

**Step 5.1: Prepare Data for Transformer**

In [ ]:
# Determine number of samples and features
num_samples, num_features = X_filtered.shape  # Correct variable name here
total_elements = num_samples * num_features
print(f"Total number of elements: {total_elements}")

# Try different sequence lengths that divide evenly
possible_sequence_lengths = [i for i in range(1, 200) if total_elements % (i * num_features) == 0]
print(f"Possible sequence lengths: {possible_sequence_lengths}")

In [ ]:
total_elements = 15452612

# Try sequence lengths from 10 to 300
for seq_len in range(10, 301):
    if total_elements % seq_len == 0:
        features = total_elements // seq_len
        print(f"✅ sequence_length: {seq_len}, num_features: {features}")


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch

def prepare_data_for_transformer(X_all, y_all, sequence_length, num_features, device='cpu'):
    num_samples = X_all.shape[0]
    total_elements = X_all.size
    
    print(f"Total number of elements: {total_elements}")
    
    if total_elements % (sequence_length * num_features) != 0:
        raise ValueError(f"Total number of elements in X_all ({total_elements}) cannot be reshaped into "
                         f"({sequence_length}, {num_features}). Please adjust the sequence_length or num_features.")
    
    num_samples_reshaped = total_elements // (sequence_length * num_features)
    print(f"Calculated number of samples: {num_samples_reshaped}")
    
    # ➕ Apply PCA before reshaping
    print("🔍 Applying PCA to reduce dimensionality...")
    X_flat = X_all.reshape(-1, X_all.shape[-1])
    max_components = min(X_flat.shape[0], X_flat.shape[1])
    pca = PCA(n_components=min(num_features, max_components))
    X_reduced = pca.fit_transform(X_flat)
    
    # Reshape to (num_samples, sequence_length, num_features)
    X_reshaped = X_reduced.reshape(num_samples_reshaped, sequence_length, num_features)
    
    # Normalize the data
    scaler = StandardScaler()
    X_reshaped = scaler.fit_transform(X_reshaped.reshape(-1, num_features)).reshape(num_samples_reshaped, sequence_length, num_features)
    
    # Convert to torch tensors
    X_tensor = torch.tensor(X_reshaped, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_all[:num_samples_reshaped], dtype=torch.long).to(device)
    
    return X_tensor, y_tensor


# --- Main Execution Block ---

# Choose a valid reshaping configuration (based on what you found)
sequence_length = 268
num_features = 57659

# Inspect shape
total_elements = X_filtered.size
print(f"Total number of elements in X_filtered: {total_elements}")
num_samples_reshaped = total_elements // (sequence_length * num_features)
print(f"Calculated number of samples: {num_samples_reshaped}")

# Run only if valid reshape
if total_elements % (sequence_length * num_features) == 0:
    print(f"✅ Reshaping works with sequence_length {sequence_length}.")
    X_tensor, y_tensor = prepare_data_for_transformer(
        X_filtered,
        y_filtered,
        sequence_length,
        num_features,
        device='mps' if torch.backends.mps.is_available() else 'cpu'
    )
    print(f"Prepared data shape for Transformer:\nX_tensor: {X_tensor.shape}, y_tensor: {y_tensor.shape}")
else:
    print(f"❌ Reshaping failed. Try adjusting the sequence length or num_features.")


**Step 5.2: Define the Transformer Model**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class EEGTransformer(nn.Module):
    def __init__(self, num_features, num_classes, sequence_length, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super(EEGTransformer, self).__init__()

        self.num_features = num_features
        self.num_classes = num_classes
        self.sequence_length = sequence_length

        # Input embedding layer to project input features to d_model
        self.embedding = nn.Linear(num_features, d_model)

        # Transformer Encoder Layer setup with batch_first=True for better performance
        self.transformer_encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(self.transformer_encoder_layer, num_layers=num_layers)

        # Fully connected layers for classification
        self.fc1 = nn.Linear(d_model * sequence_length, 512)  # Flattened transformer output
        self.fc2 = nn.Linear(512, num_classes)

        # Dropout for regularization
        self.dropout = nn.Dropout(p=dropout)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Forward pass through the model.
        x: (batch_size, sequence_length, num_features)
        """
        # Apply the embedding layer: Shape will be (batch_size, sequence_length, d_model)
        x = self.embedding(x)

        # Pass through transformer encoder layers (batch_first=True means input shape is (batch_size, sequence_length, d_model))
        x = self.transformer_encoder(x)

        # Apply LayerNorm
        x = self.layer_norm(x)

        # Flatten the transformer output to (batch_size, sequence_length * d_model)
        x = x.view(x.size(0), -1)  # Flatten: (batch_size, sequence_length * d_model)

        # Apply fully connected layers for classification
        x = F.relu(self.fc1(x))
        x = self.fc2(x)  # Output: (batch_size, num_classes)

        return x

# Hyperparameters (adjusted)
num_features = 57659  # Features per time step
num_classes = 3  # Output classes (Control and PD)
sequence_length = 268  # Sequence length per input
d_model = 256  # Reduced model dimension for regularization
nhead = 8  # Number of attention heads (divisible by d_model)
num_layers = 4  # Reduced number of layers to avoid overfitting
dropout = 0.3  # Increased dropout rate for regularization

# Instantiate the model with adjusted hyperparameters
model = EEGTransformer(num_features=num_features, num_classes=num_classes, sequence_length=sequence_length, 
                       d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)

# Print the model architecture
print("EEG Transformer Model Architecture:")
print(model)


**Step 5.4: Initialize the Model, Optimizer, and Loss Function**

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# Model, Optimizer, Loss Function
model = EEGTransformer(num_features=num_features, num_classes=num_classes, sequence_length=sequence_length, 
                       d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)

# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer and Loss function
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)  # Using AdamW with weight decay
loss_function = torch.nn.CrossEntropyLoss()

# Learning Rate Scheduler (Optional)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)  # Decay LR by 0.1 every 5 epochs



**Step 5.5: Train the Model**

In [ ]:
from tqdm import tqdm
import torch

def train(model, train_loader, optimizer, loss_function, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0
    
    # Wrapping the train_loader with tqdm for progress tracking
    for inputs, labels in tqdm(train_loader, desc="Training", unit="batch", ncols=100):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()  # Clear previous gradients
        
        # Forward pass
        outputs = model(inputs)
        
        # Calculate loss
        loss = loss_function(outputs, labels)
        running_loss += loss.item()
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Track accuracy
        _, predicted = torch.max(outputs, 1)
        correct_preds += (predicted == labels).sum().item()
        total_preds += labels.size(0)
    
    avg_loss = running_loss / len(train_loader)
    accuracy = correct_preds / total_preds

    # Optional: Print training results for each epoch
    print(f"Training Loss: {avg_loss:.4f}, Accuracy: {accuracy * 100:.2f}%")
    
    return avg_loss, accuracy


**Validate**

In [ ]:
def validate(model, val_loader, loss_function, device):
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0
    
    with torch.no_grad():  # Disable gradient computation during validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(inputs)
            
            # Calculate loss
            loss = loss_function(outputs, labels)
            running_loss += loss.item()
            
            # Track accuracy
            _, predicted = torch.max(outputs, 1)
            correct_preds += (predicted == labels).sum().item()
            total_preds += labels.size(0)
    
    # Calculate average loss and accuracy
    avg_loss = running_loss / len(val_loader)
    accuracy = correct_preds / total_preds

    # Optionally, you can print or log the results for better insight
    print(f"Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy * 100:.2f}%")
    
    return avg_loss, accuracy



**Create Dataloader**

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

def preprocess_tensor(tensor, mean=None, std=None):
    """
    Sanitize and normalize tensor:
    - Removes NaN/inf
    - Clamps extreme values
    - Normalizes using mean/std (per feature across all time steps)
    """
    tensor = torch.nan_to_num(tensor, nan=0.0, posinf=1e6, neginf=-1e6)
    tensor = torch.clamp(tensor, min=-1e3, max=1e3)

    if mean is None or std is None:
        mean = tensor.mean(dim=(0, 1), keepdim=True)
        std = tensor.std(dim=(0, 1), keepdim=True)

    tensor = (tensor - mean) / (std + 1e-6)
    return tensor, mean, std

X_train_tensor_raw = torch.tensor(X_train_np, dtype=torch.float32)
X_val_tensor_raw = torch.tensor(X_val_np, dtype=torch.float32)

X_train_tensor, train_mean, train_std = preprocess_tensor(X_train_tensor_raw)
X_val_tensor, _, _ = preprocess_tensor(X_val_tensor_raw, mean=train_mean, std=train_std)


assert X_train_filtered.shape[1] == sequence_length * num_features, \
    f"Expected shape mismatch: got {X_train_filtered.shape[1]}, expected {sequence_length * num_features}"

# Assuming X_train_filtered and X_val_filtered are the training and validation input data
# Replace the below lines with your actual data
X_train_filtered = np.random.rand(142, 603)  # Example: Replace with your actual training data
X_val_filtered = np.random.rand(30, 603)    # Example: Replace with your actual validation data

# Define sequence length and number of features
sequence_length = 57659  # Number of time steps
num_features = 268      # Number of features per time step

# ---- Reshape Input Data ----
# Reshape the features: from (n_samples, 603) to (n_samples, sequence_length, num_features)
X_train_np = X_train_filtered.reshape(-1, sequence_length, num_features)
X_val_np = X_val_filtered.reshape(-1, sequence_length, num_features)

# Verify the shape of the reshaped data
print("Reshaped input shape (train):", X_train_np.shape)  # Should be (n_samples, 67, 9)
print("Reshaped input shape (val):", X_val_np.shape)      # Should be (n_samples, 67, 9)

# ---- Sanitize and Normalize Function ----

# Apply sanitization and normalization


# Optionally, check stats
print("✅ Train tensor stats:", X_train_tensor.min().item(), X_train_tensor.max().item(), X_train_tensor.mean().item())

# ---- Generate Random Labels for Example (Replace with Actual Labels) ----
y_train_tensor = torch.tensor(np.random.randint(0, 2, size=(142,)), dtype=torch.long)  # Example: Replace with actual labels
y_val_tensor = torch.tensor(np.random.randint(0, 2, size=(30,)), dtype=torch.long)    # Example: Replace with actual labels

# ---- Create Dataset and DataLoader ----
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Verify the input shape in the DataLoader
sample_inputs, _ = next(iter(train_loader))
print("✅ Input shape to model:", sample_inputs.shape)  # Should now be [batch_size, 67, 9]

# ---- Normalize Tensor Function ----


# Normalize the data using the same mean and std for train and validation

# ---- Final Dataset and DataLoader ----
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True
)

# Verify the input shape in the DataLoader
sample_inputs, _ = next(iter(train_loader))
print("✅ Input shape to model:", sample_inputs.shape)  # Should now be [batch_size, 67, 9]


**Train for Mulitple Epochs**

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import time
import torch
import gc
from tqdm import tqdm

# ---- Configuration ----
config = {
    "batch_size": 32,
    "n_workers": 1,
    "valid_steps": 500,
    "total_steps": 5000,
}

# ---- Device setup ----
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"[Start Training] Target total steps: {config['total_steps']}")
print(f"Using device: {device}")

# ---- Move model to device ----
model.to(device)

# ---- Optimizer ----
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# ---- Class weights for 3 classes (adjust weights as needed) ----
class_weights = torch.tensor([1.0, 2.0, 1.5], dtype=torch.float32).to(device)  # Example weights

# ---- Create sample weights for WeightedRandomSampler ----
# Make sure y_train_tensor is your training labels tensor, shape [num_samples]
sample_weights = [class_weights[label.item()] for label in y_train_tensor]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_train_tensor), replacement=True)

# ---- DataLoader with sampler (no shuffle) ----
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    sampler=sampler,
    num_workers=config['n_workers'],
    pin_memory=True,
    persistent_workers=True,
)

# ---- Validation DataLoader (no sampler) ----
val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['n_workers'],
    pin_memory=True,
    persistent_workers=True,
)

# ---- Loss function with class weights ----
loss_function = torch.nn.CrossEntropyLoss(weight=class_weights).to(device)

# ---- Learning rate scheduler ----
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

# ---- Training Loop ----
step = 0
start_time = time.time()

with tqdm(total=config['total_steps'], desc="Training Progress", unit="step", miniters=100) as pbar:
    while step < config['total_steps']:
        model.train()
        running_loss = 0.0
        correct_preds = 0
        total_preds = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # Skip batch if NaNs or Infs are present
            if torch.isnan(inputs).any() or torch.isinf(inputs).any():
                continue
            if torch.isnan(labels).any() or torch.isinf(labels).any():
                continue

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(outputs, 1)
            correct_preds += (predicted == labels).sum().item()
            total_preds += labels.size(0)
            running_loss += loss.item()

            step += 1
            pbar.update(1)

            # Validation
            if step % config['valid_steps'] == 0:
                model.eval()
                val_loss = 0.0
                correct_preds_val = 0
                total_preds_val = 0

                with torch.no_grad():
                    for val_inputs, val_labels in val_loader:
                        val_inputs, val_labels = val_inputs.to(device, non_blocking=True), val_labels.to(device, non_blocking=True)

                        # Reshape validation input if needed
                        if val_inputs.dim() == 2:
                            val_inputs = val_inputs.view(-1, sequence_length, num_features)

                        val_outputs = model(val_inputs)
                        val_loss += loss_function(val_outputs, val_labels).item()
                        _, val_predicted = torch.max(val_outputs, 1)
                        correct_preds_val += (val_predicted == val_labels).sum().item()
                        total_preds_val += val_labels.size(0)

                train_accuracy = correct_preds / total_preds if total_preds > 0 else 0
                train_loss = running_loss / (batch_idx + 1)
                val_accuracy = correct_preds_val / total_preds_val if total_preds_val > 0 else 0
                val_loss_avg = val_loss / len(val_loader)

                elapsed = time.time() - start_time
                step_rate = pbar.n / elapsed if elapsed > 0 else 0

                print(f"\nTrain: {step}/{config['total_steps']} [{elapsed:.2f}s, {step_rate:.2f} step/s, "
                      f"accuracy={train_accuracy:.4f}, loss={train_loss:.4f}]")
                print(f"Valid: {step}/{config['total_steps']} [{elapsed:.2f}s, {step_rate:.2f} step/s, "
                      f"accuracy={val_accuracy:.4f}, loss={val_loss_avg:.4f}]")

                # Clear cache for MPS devices only
                if device.type == 'mps':
                    torch.mps.empty_cache()
                gc.collect()

            scheduler.step()

            if step >= config["total_steps"]:
                break

print(f"[Training Complete] Total Time: {time.time() - start_time:.2f}s")


**Evaluate**

In [ ]:
def evaluate(model, test_loader, device):
    model.eval()
    correct_preds = 0
    total_preds = 0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # ⚠️ Fix: Reshape to match model input shape
            inputs = inputs.view(-1, 67, 9)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            
            correct_preds += (predicted == labels).sum().item()
            total_preds += labels.size(0)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    
    accuracy = correct_preds / total_preds
    return accuracy, all_labels, all_preds


# Evaluate the model on the test set
test_accuracy, test_labels, test_preds = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_accuracy:.4f}")



**Visualize**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Classification Report
print(classification_report(test_labels, test_preds))

model.eval()
with torch.no_grad():
    sample_input = X_train_tensor[0].unsqueeze(0).to(device)  # Shape: (1, seq_len, features)
    sample_output = model(sample_input)
    print("✅ Sample model output:", sample_output)

In [ ]:
print("✅ Input stats:", inputs.min().item(), inputs.max().item(), inputs.mean().item(), inputs.std().item())


In [ ]:
import torch

print(torch.backends.mps.is_available())  # Should return True if MPS is available


In [ ]:
from collections import Counter
print("Train:", Counter(y_train))
print("Val:", Counter(y_val))
